# Track 10 — Capstone: 사내 지식베이스 QA (Internal KB QA)

## RAG QA와 인용(citation)이란?

사내 정책·매뉴얼처럼 모델이 학습하지 않은 지식을 다룰 때는 답변보다 **근거 확인**이 먼저입니다. 이 캡스톤은 작은 사내 KB를 대상으로 RAG QA 흐름을 끝까지 구현합니다.

- **검색:** 질문과 관련된 스니펫을 찾습니다.
- **컨텍스트 제공:** 검색된 스니펫만 모델 입력에 넣습니다.
- **답변 + 인용:** `{answer, sources}` 구조로 답하고 출처 태그를 남깁니다.
- **인용 게이트:** 인용한 출처가 실제 검색 결과 안에 있는지 검사합니다.
- **패키지:** 라이브 평가와 정적 메트릭 데모를 `_out/01/capstone_package.json`으로 저장합니다.

예시 KB는 `internal_manual_snippets.json`입니다. HR·IT·보안 매뉴얼을 줄인 샘플 8개 스니펫, 6개 출처로 구성됩니다.

## 이 노트북에서 보여줄 것

| Session | 보여주는 것 | 목적 |
|---|---|---|
| 1. Setup | `exaone` facade 초기화 · 골든셋 로더 · 정적 메트릭 데모 · 패키지 저장 헬퍼 | 공통 준비 |
| 2. 예시 KB + 검색 도구 | KB 확인 → `retrieve_snippets` · `retrieve` 도구 등록 | API 키 없이 검색 단계 실습 |
| 3. 근거 RAG QA | 검색 → 컨텍스트 → `{answer, sources}` → 인용 게이트 | 답변과 출처 검증 흐름 확인 |
| 4. 라이브 평가 + 패키지 | KB 전용 사례를 실제 RAG 경로로 평가하고 패키지 저장 | 최소 회귀 평가 흐름 |

## 이 노트북을 마치면

- 검색·컨텍스트·인용으로 이어지는 RAG QA를 직접 구현할 수 있습니다.
- 답변의 출처가 검색 결과에 포함되는지 자동으로 검사할 수 있습니다.
- 정적 fixture 메트릭 데모와 라이브 에이전트 평가를 구분할 수 있습니다.

**산출물:** `_out/01/capstone_package.json`  
**실행 조건:** Session 1·2·정적 메트릭 데모는 API 키 없이 실행됩니다. Session 3·4 라이브 평가는 EXAONE API 키가 필요합니다.

> 요약: 사내 KB 질문에 근거와 출처를 붙이고, 최소 라이브 평가까지 묶어 제출 패키지로 마감하는 캡스톤입니다.


## Session 1. Setup


### Session 1-1. Setup

**작업:** 경로, 클라이언트, 골든셋 로더, 정적 메트릭 데모, 패키지 저장 헬퍼를 준비합니다.

**정상 출력:** `exaone` 버전, `model`, `HAS_API`가 한 줄로 출력됩니다.

**의미:** 이후 검색, RAG, 평가 단계에서 사용할 공통 준비를 마칩니다.


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

import logging

# (en) Quiet library logs so the notebook output stays readable.
# (kr) 노트북 출력이 읽기 쉽도록 라이브러리 로그 수준을 낮춘다.
for _log_name in ("exaone", "exaone.llm", "exaone.llm.exaone_client", "urllib3"):
    logging.getLogger(_log_name).setLevel(logging.ERROR)

# (en) Facade-only startup; requires editable install at the repo root.
# (kr) `exaone` facade만 사용해 시작한다. 저장소 루트에서 editable 설치가 필요하다.
try:
    import exaone
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "`exaone`이 설치되지 않았습니다. 저장소 루트에서 "
        "`pip install -r requirements.txt && pip install -e ./exaone` 명령을 실행한 뒤 커널을 재시작하세요."
    ) from exc

exaone.load_project_env()
ROOT = exaone.project_root()
TRACK10 = ROOT / "recipes" / "track10_ax_capstones"
DATA = TRACK10 / "data"
SNIPPETS_PATH = ROOT / "recipes" / "track04_rag_and_knowledge" / "data" / "internal_manual_snippets.json"

API_KEY = os.environ.get("EXAONE_API_KEY", "").strip()
BASE_URL = os.environ.get("EXAONE_BASE_URL", "").strip() or "http://localhost:8000/v1"
MODEL = os.environ.get("EXAONE_MODEL", "").strip() or exaone.llm.ExaoneClient.DEFAULT_MODEL
HAS_API = bool(API_KEY)
client = None
if HAS_API:
    client = exaone.llm.ExaoneAPIClient(base_url=BASE_URL, model=MODEL, api_key=API_KEY)
print("exaone", exaone.__version__, "| model", MODEL, "| HAS_API", HAS_API)


def load_capstone_golden(tag: str) -> list[dict]:
    # (en) Load golden rows for this capstone id plus the shared "all" rows.
    # (kr) 캡스톤별 사례와 공통 "all" 사례를 함께 불러온다.
    rows: list[dict] = []
    for line in (DATA / "capstone_golden.jsonl").read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        row = json.loads(line)
        if row.get("capstone") in (tag, "all"):
            rows.append(row)
    return rows


def metric_demo_on_fixtures(rows: list[dict]) -> dict:
    # (en) Metric MECHANICS demo over STATIC golden fixtures (trial_content / expected_answer).
    #      It scores canned text, NOT the live RAG agent — see Session 4 for the real agent eval.
    # (kr) 정적 골든 fixture(trial_content / expected_answer) 기반 메트릭 계산 데모.
    #      라이브 RAG 에이전트가 아니라 미리 준비된 텍스트를 채점한다. 실제 에이전트 평가는 Session 4에서 다룬다.
    from eval.metrics import m1_task_success, m6_schema_adherence
    from eval.metrics.m1_task_success import TaskGold
    from eval.metrics.m6_schema_adherence import SchemaSpec
    from eval.metrics.m9_faithfulness import LengthRatioJudge
    from eval.metrics.types import TrialResult

    m1s, m6s, m9s, cases = [], [], [], []
    stub_judge = LengthRatioJudge()  # (en) test-only stub / (kr) 테스트 전용 스텁
    for row in rows:
        tid = row["id"]
        content = row.get("trial_content") or str(row.get("expected_answer", ""))
        tr = TrialResult(
            trial_id=f"cap-{tid}",
            task_id=tid,
            dataset="track10.golden",
            runner="fixture",
            final_content=content,
        )
        m1 = m6 = m9 = None
        if row.get("expected_answer") is not None:
            m1 = m1_task_success.score_trial_exact(tr, TaskGold(task_id=tid, answer=row["expected_answer"]))
            m1s.append(m1)
        rk = row.get("required_keys")
        if rk:
            _, loose = m6_schema_adherence.score_trial(tr, SchemaSpec(required_keys=rk))
            m6 = loose
            m6s.append(1.0 if loose else 0.0)
        if row.get("grounding_context"):
            m9 = stub_judge(trial=tr, gold={"context": row["grounding_context"]})
            m9s.append(m9)
        cases.append({"id": tid, "M1": m1, "M6": m6, "M9_stub": m9})
    mean = lambda xs: sum(xs) / len(xs) if xs else 0.0
    return {
        "n": len(rows),
        "M1_mean": mean(m1s),
        "M6_loose_mean": mean(m6s),
        "M9_stub_mean": mean(m9s),
        "m9_note": "M9=LengthRatioJudge는 테스트 전용 스텁(운영 금지)이며, 충실도가 아니라 길이 비율 근사치만 계산합니다",
        "cases": cases,
    }


def save_package(capstone_nb: str, body: dict) -> Path:
    # (en) Write capstone_package.json under <track>/_out/<nb>/ (absolute path, CWD-independent).
    # (kr) 절대경로를 사용해 <track>/_out/<nb>/capstone_package.json을 저장한다(커널 CWD와 무관).
    out_dir = TRACK10 / "_out" / capstone_nb
    out_dir.mkdir(parents=True, exist_ok=True)
    slo = exaone.observability.SLOSpec(
        name=f"capstone-{capstone_nb}",
        p95_chat_latency_ms=8000,
        structured_output_success_min="95%",
        notes="Track 10 capstone — adjust per deployment.",
    )
    payload = {
        "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "capstone_id": capstone_nb,
        "slo": slo.to_dict(),
        **body,
    }
    path = out_dir / "capstone_package.json"
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    print("saved", path.resolve())
    return path

**출력 해석:** `exaone 0.1.0 | model … | HAS_API True/False` 한 줄이 보이면 준비가 끝난 것입니다.

- `HAS_API`가 `True`이면 Session 3·4 라이브 평가가 실행되고, `False`이면 해당 단계는 건너뜁니다(Session 1·2·정적 메트릭 데모는 키 없이 실행됩니다).
- `client`와 `TRACK10`·`DATA`·`SNIPPETS_PATH`가 **절대경로**로 설정되어, 뒤의 셀을 어떤 CWD에서 실행하더라도 같은 파일을 읽고 같은 위치에 저장합니다.


## Session 2. 예시 지식베이스 + 검색 도구

**예시 지식베이스:** `internal_manual_snippets.json`은 한 조직의 HR·IT·보안 매뉴얼을 축약한 **샘플 KB**(8개 스니펫·6개 출처)입니다. 각 스니펫은 `id` · `source`(출처 태그) · `text`(본문)로 구성되며, 이후 검색·인용·평가는 모두 이 코퍼스를 기준으로 진행합니다.


### Session 2-1. 예시 지식베이스 둘러보기

**작업:** 검색과 인용의 대상이 되는 예시 KB(`internal_manual_snippets.json`)를 먼저 살펴봅니다.

**정상 출력:** `예시 KB: 사내 매뉴얼 8개 스니펫 · 출처 6종 (…)` 한 줄과 출처별 1줄 미리보기가 출력됩니다.

**의미:** 이후 단계의 전제가 되는 코퍼스를 확인하고, 무엇을 검색하고 어떤 출처를 인용할지 파악합니다.


In [ ]:
# (en) Peek at the example KB before searching it: snippet count, source tags, one line each.
# (kr) 검색하기 전에 예시 KB의 스니펫 수, 출처 태그, 1줄 미리보기를 확인한다.
snippets = json.loads(SNIPPETS_PATH.read_text(encoding="utf-8"))["snippets"]
sources = sorted({s["source"] for s in snippets})
print(f"예시 KB: 사내 매뉴얼 {len(snippets)}개 스니펫 · 출처 {len(sources)}종 ({', '.join(sources)})")
for s in snippets:
    print(f"  [{s['source']}] {s['text'][:46]}…")

**출력 해석:** 이 캡스톤의 KB는 한 조직의 HR·IT·보안 매뉴얼을 축약한 **8개 스니펫(6개 출처)**입니다.

- 각 줄 앞의 `[출처]` 태그(`hr-leave-policy`·`it-vpn-guide`·`security-phishing`·`payroll-calendar`·`benefits-education`·`security-dlp`)가 곧 **인용 단위**입니다. Session 3의 답변은 이 태그로 출처를 밝히고, 인용 게이트는 인용한 출처가 이 목록 안에 있는지 검사합니다.
- 스니펫은 `id`·`source`·`text` 구조의 작은 코퍼스라 벡터 DB 없이 키워드 검색으로도 충분합니다. 실무에서는 수천~수만 개 청크와 임베딩 검색이 필요합니다.
- 골든 평가의 KB 질문(`kb01` 연차, `kb02` VPN)도 이 코퍼스의 `hr-leave-policy`·`it-vpn-guide`를 대상으로 합니다.


### Session 2-2. 키워드 검색 + retrieve 도구

**작업:** 키워드 일치 기반 검색 함수(`retrieve_snippets`)와 `retrieve` 도구를 정의·등록하고, 데모 질의 하나를 실행합니다.

**정상 출력:** 데모 질의("연차 HR")의 검색 결과가 `[source] 본문` 형태로 출력됩니다.

**의미:** RAG의 첫 단계인 검색을 API 키 없이도 확인할 수 있습니다.


In [ ]:
# (en) Reuse the `snippets` corpus loaded in the previous cell (Session 2-1).
# (kr) 앞 셀(Session 2-1)에서 불러온 `snippets` 코퍼스를 재사용한다.
def retrieve_snippets(query: str, top_k: int = 3) -> list[dict]:
    # (en) Deterministic keyword overlap rank (no vector DB required for the capstone).
    # (kr) 키워드 겹침으로 순위를 매기는 결정론적 검색(벡터 DB 불필요).
    q = set(query.lower().split())
    scored = []
    for s in snippets:
        text = s["text"].lower()
        hit = sum(1 for w in q if w in text)
        if hit:
            scored.append((hit, s))
    scored.sort(key=lambda x: -x[0])
    return [s for _, s in scored[:top_k]]


def exec_retrieve(_name: str, args: dict) -> dict:
    # (en) Tool executor: validate the query, then return retrieved snippets as [source] text.
    # (kr) 도구 실행 함수: query를 검증한 뒤 검색된 스니펫을 [source] 본문 형태로 반환한다.
    q = (args.get("query") or "").strip()
    if not q:
        return exaone.tools.ToolResult.validation_error(source="retrieve", error="query required").to_dict()
    hits = retrieve_snippets(q)
    body = "\n".join(f"[{h['source']}] {h['text']}" for h in hits) or "(no hits)"
    return exaone.tools.ToolResult.success(content=body, source="retrieve", metadata={"hits": len(hits)}).to_dict()


RETRIEVE_SCHEMA = {
    "type": "function",
    "function": {
        "name": "retrieve",
        "description": "Search internal manual snippets",
        "parameters": {
            "type": "object",
            "required": ["query"],
            "properties": {"query": {"type": "string"}},
            "additionalProperties": False,
        },
    },
}
reg = exaone.tools.ToolRegistry()
reg.register(exaone.tools.tool_from_callable("retrieve", RETRIEVE_SCHEMA, exec_retrieve))
demo = exec_retrieve("retrieve", {"query": "연차 HR"})
print(demo.get("content", "")[:200])

**출력 해석:** "연차 HR"을 검색하면 `[hr-leave-policy] …`, `[benefits-education] …` 두 스니펫이 점수 순으로 반환됩니다.

- 출력은 청크 **id** 목록이 아니라 `[출처] 본문`입니다. 다음 단계(Session 3)에서는 이 본문을 그대로 컨텍스트로 제공합니다.
- 8개 스니펫 중 "연차"·"hr" 토큰이 일치한 스니펫만 반환되고(top_k=3 중 2개 매칭), 나머지는 0점이라 제외됩니다. 결정론적 키워드 검색이므로 동의어와 오타에는 약합니다.


## Session 3. 근거 RAG QA + 인용 게이트


### Session 3-1. 근거 RAG QA + 인용 게이트

**작업:** 검색 → 스니펫을 컨텍스트로 제공 → 구조화된 `{answer, sources}` 생성 → 인용 게이트까지 이어지는 전체 흐름을 한 번 실행해 출력합니다.

**정상 출력:** `질문 → 검색된 출처 → 답 → 인용(sources) → 인용 게이트 PASS/FAIL` 또는 API 키가 없을 때의 건너뜀 메시지가 출력됩니다.

**의미:** 모델이 검색된 근거를 바탕으로 답하고, 인용이 실제로 검색된 출처를 가리키는지 검증합니다.


In [ ]:
def _norm_sources(srcs: list) -> list[str]:
    # (en) Strip [] brackets/whitespace and dedup cited source tags for comparison.
    # (kr) 인용 출처 태그에서 [] 괄호와 공백을 제거하고, 중복을 없애 비교용으로 정규화한다.
    out: list[str] = []
    for s in srcs or []:
        tag = str(s).strip().strip("[]").strip()
        if tag and tag not in out:
            out.append(tag)
    return out


def citation_gate(retrieved: list[str], cited: list[str]) -> bool:
    # (en) Citation gate: cited sources must be non-empty AND a subset of the retrieved sources.
    # (kr) 인용 게이트: 인용 출처가 비어 있지 않고 실제로 검색된 출처의 부분집합인지 확인한다.
    return bool(cited) and all(c in retrieved for c in cited)


def grounded_rag_answer(question: str, top_k: int = 3) -> dict:
    # (en) One explicit RAG cycle: retrieve -> inject snippets as context -> structured {answer, sources}
    #      -> citation gate (cited sources must be a subset of the retrieved sources). English system prompt.
    # (kr) 명시적인 RAG 한 사이클: 검색 -> 스니펫을 컨텍스트로 제공 -> 구조화된 {answer, sources} 생성
    #      -> 인용 게이트(인용 출처가 검색된 출처 안에 있어야 함). 영어 system prompt를 사용한다.
    hits = retrieve_snippets(question, top_k=top_k)
    retrieved: list[str] = []
    for h in hits:
        if h["source"] not in retrieved:
            retrieved.append(h["source"])
    context = "\n".join(f"[{h['source']}] {h['text']}" for h in hits) or "(no hits)"
    system = (
        "You are an internal policy (sample) assistant. Answer using ONLY the provided context snippets, "
        "in Korean polite style (합니다체). Output a JSON object with keys `answer` (string) and `sources` "
        "(array of the [source] tags you actually used). Cite only tags present in the context; if the "
        "context does not contain the answer, set answer to a brief refusal and sources to []."
    )
    user = f"# Context\n{context}\n\n# Question\n{question}"
    resp = client.chat([
        exaone.llm.ExaoneMessage(role="system", content=system),
        exaone.llm.ExaoneMessage(role="user", content=user),
    ])
    parsed = exaone.output.StructuredOutputPipeline(required_keys=["answer", "sources"]).process(resp.content or "")
    data = parsed.data if parsed.success and isinstance(parsed.data, dict) else {"answer": (resp.content or "")[:300], "sources": []}
    cited = _norm_sources(data.get("sources"))
    return {
        "question": question,
        "retrieved": retrieved,
        "answer": data.get("answer", ""),
        "sources": cited,
        "citation_ok": citation_gate(retrieved, cited),
    }


live_answer = None
trace = [{"event": "capstone_start", "capstone": "01"}]
if HAS_API:
    QUESTION = "연차는 어디서 신청하나요?"
    live_answer = grounded_rag_answer(QUESTION)
    print("질문:", QUESTION)
    print("검색된 출처:", live_answer["retrieved"])
    print("답변:", live_answer["answer"])
    print("인용(sources):", live_answer["sources"])
    print("인용 게이트:", "PASS ✅ (sources ⊆ 검색된 출처)" if live_answer["citation_ok"] else "FAIL ❌ (출처 없음/날조 인용)")
    trace.append({
        "event": "qa",
        "question": QUESTION,
        "retrieved": live_answer["retrieved"],
        "sources": live_answer["sources"],
        "citation_ok": live_answer["citation_ok"],
    })
else:
    print("(skip) EXAONE API 키가 없어 라이브 RAG QA를 건너뜁니다. Session 1·2·정적 메트릭 데모는 키 없이 실행됩니다.")
    trace.append({"event": "qa_skipped", "reason": "no API key"})

**출력 해석:** API 키가 있으면 예시 질문 "연차는 어디서 신청하나요?"에 대해 `[hr-leave-policy]`가 검색되고, 답변의 `sources`가 해당 출처를 가리키면 인용 게이트가 **PASS**가 됩니다.

- 이상적인 실행에서는 "연차는 HR 포털 > 휴가에서 신청합니다."처럼 검색된 스니펫 문구에 근거한 답변이 나옵니다(컨텍스트 제공 효과).
- 인용 게이트는 `sources ⊆ 검색된 출처`만 확인하는 **인용 무결성** 검사입니다. 검색되지 않은 id를 인용하거나 `sources`가 비어 있으면 **FAIL**로 잡아냅니다. 다만 이는 출처 태그가 실제 검색 결과에 있는지만 확인할 뿐, **답변 문장이 그 출처에 실제로 부합하는지(충실도)**는 평가하지 않습니다(태그만 맞추면 PASS될 수 있습니다). 답변의 근거성은 Session 4의 `근거 통과율`(답↔근거 겹침)로 확인하고, 운영 환경에서는 별도 faithfulness judge로 평가해야 합니다.
- API 키가 없으면 `(skip)` 메시지가 출력되고 라이브 단계는 건너뜁니다. 응답은 비결정적 샘플링을 사용하므로 문장 표현은 실행마다 달라질 수 있지만, 게이트는 인용의 구조적 유효성을 항상 검사합니다.


### Session 3-2. 인용 게이트가 잡아내는 것 (반례)

**작업:** 같은 검색 결과(`['hr-leave-policy']`)에 정상 인용, 날조 인용, 빈 인용 세 가지를 같은 `citation_gate`에 넣어 PASS/FAIL을 나란히 비교합니다(키 불필요·결정론적).

**정상 출력:** `[PASS ✅] 정상 인용`, `[FAIL ❌] 날조 인용`, `[FAIL ❌] 빈 인용` 세 줄이 출력됩니다.

**의미:** 게이트가 "항상 PASS"를 반환하지 않고 무엇을 걸러내는지 보여줍니다. 그래서 Session 4의 `인용 통과율`이 의미를 갖습니다.


In [ ]:
# (en) The gate is NOT "always PASS": run the SAME citation_gate on crafted cases to see what trips it.
#      Deterministic and key-free — no model call needed.
# (kr) 게이트는 "항상 PASS"가 아니다. 같은 citation_gate에 직접 만든 사례를 넣어 무엇이 걸러지는지 확인한다.
#      모델 호출 없이 결정론적으로 실행되며 API 키가 필요 없다.
retrieved_demo = ["hr-leave-policy"]  # (en) sources actually retrieved / (kr) 실제 검색된 출처
gate_examples = [
    ("정상 인용 (검색된 출처)", ["hr-leave-policy"]),
    ("날조 인용 (검색되지 않은 출처)", ["it-vpn-guide"]),
    ("빈 인용 (sources 비어 있음)", []),
]
for label, cited in gate_examples:
    verdict = "PASS ✅" if citation_gate(retrieved_demo, cited) else "FAIL ❌"
    print(f"  [{verdict}] {label}: sources={cited}  (검색된 출처={retrieved_demo})")

**출력 해석:** 같은 검색 결과 `['hr-leave-policy']`에 인용 목록만 바꿔 넣으면 게이트 판정이 갈립니다.

- **정상 인용** `['hr-leave-policy']` → **PASS**: 인용이 실제로 검색된 출처와 일치합니다.
- **날조 인용** `['it-vpn-guide']` → **FAIL**: 검색되지 않은 출처를 인용했습니다. 이 캡스톤이 막으려는, 그럴듯하지만 근거 없는 인용입니다.
- **빈 인용** `[]` → **FAIL**: 출처 없이 단정한 답변입니다.
- 따라서 Session 4의 `인용 통과율 1.00`은 "검사를 안 한 것"이 아니라 "위 FAIL 사례가 없다"는 뜻입니다.


## Session 4. 라이브 KB 평가 + 패키지


### Session 4-1. 라이브 KB 평가 + 패키지

**작업:** KB 골든셋 사례를 실제 RAG 경로로 실행해 평가(인용·근거 통과율)하고, 정적 메트릭 데모를 별도로 담아 캡스톤 패키지를 저장합니다.

**정상 출력:** `라이브 KB 평가(최소) … 통과율`, `정적 메트릭 데모(fixture) …`, `saved … capstone_package.json`이 출력됩니다.

**의미:** 구현한 에이전트를 실제 RAG 경로로 실행하는 **최소 평가 흐름**과 메트릭 계산 데모를 한 파일로 묶습니다. 여기서는 2건과 간단한 근거 판정 휴리스틱만 사용합니다.


In [ ]:
all_rows = load_capstone_golden("01")
kb_rows = [r for r in all_rows if r.get("capstone") == "01"]  # (en) KB-specific rows only / (kr) KB 전용 사례만

# (en) LIVE eval SKELETON: run the SAME grounded-RAG path on this capstone's KB rows and score citation
#      integrity + a COARSE grounding heuristic. Minimal regression baseline (only 2 rows here) — for
#      production QA, scale up the rows and swap the grounding check for a real faithfulness judge.
# (kr) 라이브 평가 최소 흐름: 이 캡스톤의 KB 사례를 같은 근거 RAG 경로로 실행하고, 인용 무결성과 간단한 근거 판정 휴리스틱을 채점한다.
#      여기서는 2건의 최소 회귀 기준선만 다루므로, 운영 품질 검증에는 골든셋 사례 보강과 실제 faithfulness judge가 필요하다.
live_eval = None
if HAS_API:
    cases = []
    for r in kb_rows:
        res = grounded_rag_answer(r["query"])
        exp = r.get("expected_answer")
        # (en) Grounding: expected_answer substring, else coarse >=0.3 token overlap (heuristic, not a judge).
        # (kr) 근거 판정: expected_answer 포함 여부를 먼저 보고, 없으면 토큰 겹침 >=0.3을 적용한다(간단한 휴리스틱, judge 아님).
        if isinstance(exp, str) and exp:
            grounded = exp in res["answer"]
        else:
            ctx_toks = set((r.get("grounding_context") or "").lower().split())
            ans_toks = set(res["answer"].lower().split())
            grounded = bool(ctx_toks) and len(ctx_toks & ans_toks) / len(ctx_toks) >= 0.3
        cases.append({
            "id": r["id"],
            "query": r["query"],
            "retrieved": res["retrieved"],
            "sources": res["sources"],
            "citation_ok": res["citation_ok"],
            "answer_grounded": grounded,
        })
    n = len(cases)
    live_eval = {
        "n": n,
        "citation_pass_rate": sum(c["citation_ok"] for c in cases) / n if n else 0.0,
        "grounding_pass_rate": sum(c["answer_grounded"] for c in cases) / n if n else 0.0,
        "cases": cases,
    }
    print(f"라이브 KB 평가(최소): n={n} | 인용 통과율={live_eval['citation_pass_rate']:.2f} | 근거 통과율={live_eval['grounding_pass_rate']:.2f}")
    for c in cases:
        print(f"  - {c['id']}: cite={'OK' if c['citation_ok'] else 'X'} ground={'OK' if c['answer_grounded'] else 'X'} | {c['query']} -> {c['sources']}")
    trace.append({"event": "live_eval", "n": n, "citation_pass_rate": live_eval["citation_pass_rate"]})
else:
    print("(skip) 라이브 KB 평가는 EXAONE API 키가 필요합니다.")

# (en) Static fixture metric demo — metric MECHANICS only, NOT the live agent's performance.
# (kr) 정적 fixture 메트릭 데모: 메트릭 계산 방식만 보여주며, 라이브 에이전트 성능을 뜻하지 않는다.
metric_demo = metric_demo_on_fixtures(all_rows)
print(f"정적 메트릭 데모(fixture): n={metric_demo['n']} | M1={metric_demo['M1_mean']:.2f} M6={metric_demo['M6_loose_mean']:.2f} M9(stub)={metric_demo['M9_stub_mean']:.2f}")
print("  ↳", metric_demo["m9_note"])

assert metric_demo["n"] >= 2, "골든셋 로더가 사례를 읽지 못했습니다"
assert kb_rows, "capstone='01' KB 전용 사례가 없습니다"

pkg_path = save_package("01", {
    "index": {"n_snippets": len(snippets)},
    "demo_retrieve": demo,
    "live_answer": live_answer,
    "live_eval": live_eval,
    "metric_demo_on_fixtures": metric_demo,
    "session_trace": trace,
})

**출력 해석:** `_out/01/capstone_package.json`이 저장되면 이 캡스톤 실행은 완료됩니다.

- **라이브 KB 평가(최소)**: `kb01`·`kb02`를 실제 검색→인용 경로로 실행해 `인용 통과율`(출처 무결성)과 `근거 통과율`(답↔근거 토큰 겹침)을 계산합니다. 정적 fixture 채점과 달리 **구현한 에이전트를 실제로 실행한** 회귀 신호입니다. 다만 **2건 규모 + 간단한 토큰 겹침(≥0.3) 휴리스틱**이므로 품질 보증 지표가 아니라 **평가 흐름을 보여주는 최소 회귀 기준선**입니다. 운영 환경에서는 평가 사례 수를 늘리고 근거 판정을 LLM faithfulness judge로 바꿔야 합니다.
- **정적 메트릭 데모**: `M1/M6/M9`는 미리 준비된 fixture를 채점하는 **메트릭 계산 예시**일 뿐 에이전트 성능이 아닙니다. 특히 `M9`는 `LengthRatioJudge`(테스트 전용 스텁·운영 금지)라 충실도를 측정하지 않고 길이 비율만 근사합니다.
- 두 값을 같은 의미로 해석하면 안 됩니다. 라이브 평가(최소)는 **에이전트 실행 결과**를, 정적 데모는 **메트릭 계산 방식**을 보여줍니다.


## 마무리

이 캡스톤에서는 사내 KB QA를 **검색 → 컨텍스트 제공 → 인용 → 검증** 흐름으로 구현하고, 결과를 `_out/01/capstone_package.json`으로 저장했습니다.

**핵심 정리**
- **근거 RAG:** 질문을 검색해 얻은 스니펫만 컨텍스트로 제공하므로, 모델이 일반 지식으로 추측하는 위험을 줄입니다.
- **인용 게이트:** `sources`가 검색된 출처의 부분집합인지 검사해 빈 출처나 날조된 출처 태그를 잡습니다.
- **평가 구분:** 정적 fixture 메트릭(M1/M6/M9)은 계산 방식 데모이고, 라이브 평가는 실제 RAG 경로의 실행 결과입니다.

**한계**
- 라이브 평가는 `kb01`·`kb02` 2건과 간단한 토큰 겹침 휴리스틱만 사용합니다. 운영 지표로 쓰려면 골든셋과 faithfulness judge를 보강해야 합니다.
- 검색은 키워드 일치 방식이라 동의어·오타에 약합니다. 임베딩 검색은 Track 04를 참고하세요.
- 인용 게이트는 출처 태그의 무결성만 확인합니다. 답변 내용의 충실도는 별도 평가가 필요합니다.
- 응답은 비결정적 샘플링을 사용하므로 문장과 통과율은 실행마다 달라질 수 있습니다.

**다음:** `10_02` 회의록 → 액션아이템. 운영화는 `10_07` 프로덕션 하네스를 참고하세요.

## 체크포인트

- [ ] Session 2 `retrieve` 데모 통과 (`[source] 본문` 출력)
- [ ] Session 3 인용 게이트 PASS (`sources ⊆ 검색된 출처`)
- [ ] Session 4 `_out/01/capstone_package.json` 저장
